# Colab L4 ECG training notebook

This notebook is the Colab-ready version of the reviewer-safe ECG training pipeline. It is designed for Google Colab with an L4 GPU and follows the same fixed record-level split used in the reviewer notebook:

- QTDB: 84 train / 21 validation
- LUDB: 20 adaptation / 180 untouched test
- normalization computed from QTDB training windows only
- all heavy runs should be executed in Colab, not locally

Use this notebook in Colab, mount Google Drive, and run cells from top to bottom.

In [ ]:
%pip -q install wfdb scipy scikit-learn pandas numpy torch

from pathlib import Path
import json
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from scipy.signal import butter, find_peaks, resample_poly, sosfiltfilt
import wfdb
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/PQRST_Reviewer2')
DATA_ROOT = DRIVE_ROOT / 'datasets'
QTDB_DIR = DATA_ROOT / 'qtdb-1.0.0'
LUDB_DIR = DATA_ROOT / 'ludb-1.0.1'
ARTIFACT_DIR = DRIVE_ROOT / 'reviewer2_artifacts'
CHECKPOINT_DIR = ARTIFACT_DIR / 'checkpoints'
STATE_DIR = ARTIFACT_DIR / 'state'
for folder in [DATA_ROOT, ARTIFACT_DIR, CHECKPOINT_DIR, STATE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('GPU available:', torch.cuda.is_available())
print('Device:', DEVICE)
print('Drive root:', DRIVE_ROOT)

In [ ]:
def paired_records(folder):
    headers = {path.stem for path in Path(folder).glob('*.hea')}
    signals = {path.stem for path in Path(folder).glob('*.dat')}
    return sorted(headers & signals)

def download_once(database, target, expected_count):
    target = Path(target)
    target.mkdir(parents=True, exist_ok=True)
    complete_file = target / '.download_complete.json'
    records = paired_records(target)
    if complete_file.exists() and len(records) == expected_count:
        print(database, 'already present:', len(records), 'records')
        return records
    print('Downloading', database, 'to', target)
    wfdb.dl_database(database, dl_dir=str(target), keep_subdirs=False)
    records = paired_records(target)
    if len(records) != expected_count:
        raise RuntimeError(f'{database}: expected {expected_count} paired records, found {len(records)}')
    complete_file.write_text(json.dumps({'database': database, 'records': records, 'time': time.time()}, indent=2))
    return records

qtdb_records = download_once('qtdb', QTDB_DIR, 105)
ludb_records = download_once('ludb', LUDB_DIR, 200)
print('QTDB records:', len(qtdb_records))
print('LUDB records:', len(ludb_records))

In [ ]:
PRE = 120
POST = 240
R5_POST = 320
BATCH_SIZE = 64
NUM_EPOCHS = 30
R6_ADAPT_EPOCHS = 8
TRAINING_SEEDS = [1, 2, 3, 4, 5]

def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_global_seed(7)
print('Seed control ready.')

In [ ]:
def generate_labels(annotation, length, sample_scale=1.0):
    labels = np.zeros(length, dtype=np.int64)
    start = None
    wave_kind = None
    for sample, symbol in zip(annotation.sample, annotation.symbol):
        sample = int(round(sample * sample_scale))
        if symbol == '(':
            start = sample
            wave_kind = None
        elif symbol == 'p':
            wave_kind = 1
        elif symbol == 'N':
            wave_kind = 2
        elif symbol == 't':
            wave_kind = 3
        elif symbol == ')' and start is not None and wave_kind is not None:
            labels[max(0, start):min(length, sample + 1)] = wave_kind
            start = None
            wave_kind = None
    labels[labels == 2] = 0
    labels[labels == 3] = 2
    return labels

def pan_tompkins_r_peaks(ecg, fs):
    nyquist = fs / 2.0
    sos = butter(3, [5.0 / nyquist, 18.0 / nyquist], btype='bandpass', output='sos')
    bandpassed = sosfiltfilt(sos, ecg)
    derivative = np.convolve(bandpassed, np.array([-1, -2, 0, 2, 1]) * fs / 8.0, mode='same')
    width = max(1, round(0.150 * fs))
    integrated = np.convolve(derivative ** 2, np.ones(width) / width, mode='same')
    candidates, _ = find_peaks(integrated, distance=max(1, round(0.20 * fs)))
    boot = candidates[candidates < min(len(ecg), round(2 * fs))]
    spki = np.percentile(integrated[boot], 90) if len(boot) else 0.0
    npki = np.percentile(integrated[boot], 25) if len(boot) else 0.0
    accepted = []
    for peak in candidates:
        threshold = npki + 0.25 * (spki - npki)
        if integrated[peak] >= threshold:
            accepted.append(peak)
            spki = 0.125 * integrated[peak] + 0.875 * spki
        else:
            npki = 0.125 * integrated[peak] + 0.875 * npki
    search = round(0.10 * fs)
    refined = []
    for peak in accepted:
        left = max(0, peak - search)
        right = min(len(ecg), peak + search + 1)
        refined.append(left + int(np.argmax(ecg[left:right])))
    return np.unique(np.asarray(refined, dtype=int))

def create_windows(ecg, labels, r_peaks, post):
    xs, ys = [], []
    for r in r_peaks:
        left, right = r - PRE, r + post
        if left >= 0 and right <= len(ecg):
            xs.append(ecg[left:right])
            ys.append(labels[left:right])
    return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.int64)

def qtdb_record(record_name, post):
    path = str(QTDB_DIR / record_name)
    record = wfdb.rdrecord(path)
    ecg = record.p_signal[:, 0].astype(np.float32)
    labels = generate_labels(wfdb.rdann(path, 'pu0'), len(ecg))
    return create_windows(ecg, labels, pan_tompkins_r_peaks(ecg, float(record.fs)), post)

def ludb_record(record_name, post):
    path = str(LUDB_DIR / record_name)
    record = wfdb.rdrecord(path)
    lead = record.sig_name.index('ii')
    ecg = resample_poly(record.p_signal[:, lead].astype(np.float32), up=1, down=2)
    labels = generate_labels(wfdb.rdann(path, 'ii'), len(ecg), sample_scale=0.5)
    size = min(len(ecg), len(labels))
    ecg, labels = ecg[:size], labels[:size]
    return create_windows(ecg, labels, pan_tompkins_r_peaks(ecg, 250.0), post)

def build_partition(record_names, builder, post):
    xs, ys, ids = [], [], []
    skipped = []
    for record_name in record_names:
        try:
            x, y = builder(record_name, post)
            if len(x):
                xs.append(x)
                ys.append(y)
                ids.extend([record_name] * len(x))
        except Exception as exc:
            skipped.append({'record': record_name, 'error': str(exc)})
    if not xs:
        raise RuntimeError('No usable windows were generated.')
    return np.concatenate(xs), np.concatenate(ys), np.asarray(ids), skipped

In [ ]:
qtdb_train_records, qtdb_val_records = train_test_split(
    qtdb_records, test_size=0.20, random_state=7, shuffle=True
)
qtdb_train_records = sorted(qtdb_train_records)
qtdb_val_records = sorted(qtdb_val_records)

r6_adapt_records, r6_test_records = train_test_split(
    ludb_records, test_size=0.90, random_state=17, shuffle=True
)
r6_adapt_records = sorted(r6_adapt_records)
r6_test_records = sorted(r6_test_records)

assert len(qtdb_train_records) == 84 and len(qtdb_val_records) == 21
assert len(r6_adapt_records) == 20 and len(r6_test_records) == 180
assert set(qtdb_train_records).isdisjoint(qtdb_val_records)
assert set(r6_adapt_records).isdisjoint(r6_test_records)

X_qt_train, Y_qt_train, qt_train_ids, _ = build_partition(qtdb_train_records, qtdb_record, POST)
X_qt_val, Y_qt_val, qt_val_ids, _ = build_partition(qtdb_val_records, qtdb_record, POST)

train_mean = float(X_qt_train.mean())
train_std = float(X_qt_train.std())
if train_std == 0:
    raise ValueError('QTDB training std is zero')

X_qt_train = (X_qt_train - train_mean) / train_std
X_qt_val = (X_qt_val - train_mean) / train_std

X_lu_adapt_240, Y_lu_adapt_240, lu_adapt_ids_240, _ = build_partition(r6_adapt_records, ludb_record, POST)
X_lu_test_240, Y_lu_test_240, lu_test_ids_240, _ = build_partition(r6_test_records, ludb_record, POST)
X_lu_adapt_240 = (X_lu_adapt_240 - train_mean) / train_std
X_lu_test_240 = (X_lu_test_240 - train_mean) / train_std

print('QTDB: 84 train / 21 validation')
print('LUDB: 20 adaptation / 180 test')
print('Normalization mean/std:', train_mean, train_std)
print('Train windows:', len(X_qt_train), 'Val windows:', len(X_qt_val))
print('LUDB adapt/test windows:', len(X_lu_adapt_240), len(X_lu_test_240))

split_manifest = {
    'qtdb_train_records': qtdb_train_records,
    'qtdb_validation_records': qtdb_val_records,
    'ludb_adaptation_records': r6_adapt_records,
    'ludb_test_records': r6_test_records,
    'qtdb_train_mean': train_mean,
    'qtdb_train_std': train_std,
    'qtdb_split_seed': 7,
    'ludb_split_seed': 17,
}
(ARTIFACT_DIR / 'split_manifest.json').write_text(json.dumps(split_manifest, indent=2))
print('Saved split manifest to Drive:', ARTIFACT_DIR / 'split_manifest.json')

In [ ]:
class CNNFeatureExtractor(nn.Module):
    def __init__(self, channels=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.features(x)

class BiLSTMBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(64, 128, num_layers=1, batch_first=True, bidirectional=True)

    def forward(self, x):
        return self.lstm(x)[0]

class RPeakGuidedML2(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.cnn = CNNFeatureExtractor()
        self.bilstm = BiLSTMBlock()
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)
        return self.classifier(self.dropout(self.bilstm(x)))

class RPeakTimeML2(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(2, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        )
        self.bilstm = nn.LSTM(64, 128, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)
        return self.classifier(self.dropout(self.bilstm(x)[0]))

def time_channel(signals, post):
    time = (np.arange(signals.shape[1], dtype=np.float32) - PRE) / float(post)
    return np.stack([signals.astype(np.float32), np.broadcast_to(time, signals.shape)], axis=1).copy()

print('Model definitions ready.')

In [ ]:
def make_loader(features, labels, seed, shuffle):
    generator = torch.Generator()
    generator.manual_seed(seed)
    dataset = torch.utils.data.TensorDataset(
        torch.as_tensor(features, dtype=torch.float32),
        torch.as_tensor(labels, dtype=torch.long),
    )
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
    )

def run_training_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    batches = 0
    with torch.set_grad_enabled(training):
        for features, labels in loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(features).permute(0, 2, 1)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            total_loss += float(loss.item())
            batches += 1
    return total_loss / max(1, batches)

def predict(model, features):
    model.eval()
    preds = []
    with torch.inference_mode():
        for start in range(0, len(features), BATCH_SIZE):
            batch = torch.as_tensor(features[start:start + BATCH_SIZE], dtype=torch.float32, device=DEVICE)
            preds.append(model(batch).argmax(2).cpu().numpy())
    return np.concatenate(preds)

print('Training helpers ready.')

In [ ]:
def train_one_seed(seed, model_name='R3'):
    set_global_seed(seed)
    if model_name == 'R3':
        model = RPeakGuidedML2().to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        train_loader = make_loader(X_qt_train[:, None], Y_qt_train, seed, shuffle=True)
        val_loader = make_loader(X_qt_val[:, None], Y_qt_val, seed, shuffle=False)
    else:
        raise ValueError(f'Unsupported model_name: {model_name}')

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    best_loss = float('inf')
    best_state = None
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = run_training_epoch(model, train_loader, criterion, optimizer)
        val_loss = run_training_epoch(model, val_loader, criterion)
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f'{model_name} seed={seed} epoch={epoch:02d} train={train_loss:.4f} val={val_loss:.4f}')
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), CHECKPOINT_DIR / f'{model_name}_seed_{seed}.pth')
    return model

# Example single-seed run. Keep this as a starter; expand with the full five-seed sweep as needed.
example_model = train_one_seed(seed=1, model_name='R3')
pred = predict(example_model, X_lu_test_240[:, None])
print('LUDB test prediction shape:', pred.shape)